In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [ ]:
def qrng(n):
    backend = BasicSimulator()
    bits = []
    while len(bits) < n:
        batch = min(20, n - len(bits))
        qc = QuantumCircuit(batch, batch)
        for i in range(batch):
            qc.h(i)
        qc.measure(range(batch), range(batch))
        job = backend.run(transpile(qc, backend), shots=1)
        counts = job.result().get_counts()
        result = list(counts.keys())[0][::-1]
        bits += [int(b) for b in result]
    return bits

N = 50

print("testing qrng:", qrng(8))

In [ ]:
alice_bits = qrng(N)
alice_bases = qrng(N)

print("Alice's bits: ", alice_bits)
print("Alice's bases:", alice_bases)

In [ ]:
bob_bases = qrng(N)

qc = QuantumCircuit(N, N)

for i in range(N):
    if alice_bits[i] == 1:
        qc.x(i)
    if alice_bases[i] == 1:
        qc.h(i)

for i in range(N):
    if bob_bases[i] == 1:
        qc.h(i)

qc.measure(range(N), range(N))

backend = BasicSimulator()
job = backend.run(transpile(qc, backend), shots=1)
bob_bits = list(job.result().get_counts().keys())[0][::-1]
bob_bits = [int(b) for b in bob_bits]

print("Bob's bases:", bob_bases)
print("Bob's bits: ", bob_bits)

In [ ]:
alice_key = []
bob_key = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

print("Alice's key:", alice_key)
print("Bob's key:  ", bob_key)
print("Key length:", len(alice_key), "out of", N, "qubits")

In [ ]:
errors = 0
for i in range(len(alice_key)):
    if alice_key[i] != bob_key[i]:
        errors += 1

print("Number of errors:", errors)
print("Keys match:", alice_key == bob_key)